# Entity Resolution (Splink)

This notebook explores how entiteies can be assigned to various lines of data

In [1]:
import os
import re
import random
import uuid

import numpy as np
import pandas as pd
from faker import Faker
import pycountry
import usaddress
from address_parser import parser
from IPython.display import display

import splink
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.backends.duckdb import DuckDBAPI
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.blocking_rule_library import block_on
from splink.exploratory import profile_columns

print(splink.__version__)

4.0.16


This needs to be >3.X otherwise DuckDBAPI does not work

# Resolving Entities

#### Dunder names for methods

In [2]:
__current_data_folder__ = "" ## sets current data folder
__low_transactions__ = 2 ## low number of generated bank transactions
__high_transactions__ = 7 ## high number of generated bank transactions

#### Various maps and lists used in data cleaning steps (non PEP 8)

In [3]:
### MAPS
COUNTRY_MAPPING = {"UNITED STATES OF AMERICA": "US","UNITED STATES": "US","AMERICA": "US"," USA": "US","U S A": "US","UNITED KINGDOM": "GB", "UK": "GB", "U K": "GB", "BRITAIN": "GB", "ENGLAND": "GB", "SCOTLAND": "GB", "WALES": "GB", "NORTHERN IRELAND": "GB","CANADA": "CA", "CA": "CA","MEXICO": "MX", "MEX": "MX","AUSTRALIA": "AU", "AUS": "AU","NEW ZEALAND": "NZ", "N Z": "NZ", "NZ": "NZ","JAPAN": "JP", "JPN": "JP", "NIPPON": "JP","KOREA REPUBLIC OF": "KR", "SOUTH KOREA": "KR", "KOREA SOUTH": "KR", "REPUBLIC OF KOREA": "KR", "KOREA": "KR","CHILE": "CL","COLOMBIA": "CO","COSTA RICA": "CR","GERMANY": "DE", "DEUTSCHLAND": "DE","FRANCE": "FR", " FRA ": "FR","ITALY": "IT", "ITALIA": "IT","SPAIN": "ES", "ESPANA": "ES","PORTUGAL": "PT", " POR": "PT","IRELAND": "IE", " EIRE": "IE","NETHERLANDS": "NL", "HOLLAND": "NL", "NEDERLAND": "NL","BELGIUM": "BE", "BELGIE": "BE", "BELGIQUE": "BE","LUXEMBOURG": "LU","SWITZERLAND": "CH", "SCHWEIZ": "CH", "SUISSE": "CH", "SVIZZERA": "CH","AUSTRIA": "AT", "OSTERREICH": "AT", "ÖSTERREICH": "AT","SWEDEN": "SE", "SVERIGE": "SE","NORWAY": "NO", "NORGES": "NO", "NORGE": "NO", "NO": "NO","DENMARK": "DK", "DANMARK": "DK","FINLAND": "FI", "SUOMI": "FI","ICELAND": "IS","GREECE": "GR", "HELLAS": "GR", "ELLADA": "GR","TURKEY": "TR", "TURKIYE": "TR","ESTONIA": "EE", "EESTI": "EE","LATVIA": "LV", "LATVIJA": "LV","LITHUANIA": "LT", "LIETUVA": "LT","CZECH REPUBLIC": "CZ", "CZECHIA": "CZ", "CESKO": "CZ","SLOVAKIA": "SK", "SLOVENSKO": "SK","POLAND": "PL", "POLSKA": "PL","SLOVENIA": "SI", "SLOVENIJA": "SI","HUNGARY": "HU", "MAGYARORSZAG": "HU", "MAGYARORSZÁG": "HU","CAYMAN ISLANDS": "KY", "CAYMAN": "KY", "BERMUDA": "BM", "BRITISH VIRGIN ISLANDS": "VG", "BVI": "VG", "VIRGIN ISLANDS BRITISH": "VG", "VIRGIN ISLANDS UK": "VG", "JERSEY": "JE", "GUERNSEY": "GG", "ISLE OF MAN": "IM", "MAN ISLAND": "IM", "GIBRALTAR": "GI", "BAHAMAS": "BS", "BARBADOS": "BB", "SEYCHELLES": "SC", "MAURITIUS": "MU", "PANAMA": "PA", "LIECHTENSTEIN": "LI", "MONACO": "MC", "ANDORRA": "AD", "ANGUILLA": "AI", "ARUBA": "AW", "SAMOA": "WS", "VANUATU": "VU", "ST KITTS": "KN", "SAINT KITTS": "KN", "ST VINCENT": "VC", "SAINT VINCENT": "VC", "ST LUCIA": "LC", "SAINT LUCIA": "LC", "ANTIGUA": "AG", "BELIZE": "BZ", "COOK ISLANDS": "CK", "NETHERLANDS ANTILLES": "AN", "CURACAO": "CW", "BVI": "VG", "ISRAEL": "IL", "ISR": "IL", "SINGAPORE": "SG", "HONG KONG": "HK", "CHINA": "CN", "INDIA": "IN", "RUSSIA": "RU",}
COMPANY_MONIKERS = {"LIMITED": "LTD", "LIMTED": "LTD", "LTD.": "LTD", "LTD": "LTD", "CORPORATION": "CORP", "CORP.": "CORP", "CORP": "CORP", "INCORPORATED": "INC", "INCORPORATD": "INC", "INC.": "INC", "INC": "INC", "COMPANY": "CO", "CO.": "CO", "CO": "CO", "PROPRIETARY LIMITED": "PTY LTD", "PROPRIETARY": "PTY", "PTY LTD": "PTY LTD", "PTY. LTD.": "PTY LTD", "PT": "PT", "PTE": "PTE", "PTE LTD": "PTE LTD", "UNLIMITED": "UL", "UNLTD": "UL", "UL": "UL", "SNC": "SNC", "KD": "KD", "SPA": "SPA", "PLC": "PLC", "L.P.": "LP", "G.P.": "GP", "LLC": "LLC", "LLC.": "LLC", "LIMITED LIABILITY COMPANY": "LLC", "LIMITED LIABILITY CO": "LLC", "LIMITED LIABILITY PARTNERSHIP": "LLP", "LLP": "LLP", "LLP.": "LLP", "PRIVATE LIMITED": "PVT LTD", "PVT LTD": "PVT LTD", "PVT. LTD.": "PVT LTD","LIMITED PARTNERSHIP": "LP", "LTD PARTNERSHIP": "LP", "LP": "LP", "GENERAL PARTNERSHIP": "GP", "GP": "GP", "LP.": "LP", "GP.": "GP", "PARTNERSHIP": "PARTNERSHIP","TRUST": "TRUST", "FUND": "FUND", "FOUNDATION": "FOUNDATION","SOCIETE ANONYME": "SA", "SOCIEDAD ANONIMA": "SA", "SOCIÉTÉ ANONYME": "SA", "SOCIETA ANONIMA": "SA", "SOCIETE ANONYME A RESPONSABILITE LIMITEE": "SARL", "SOCIÉTÉ ANONYME À RESPONSABILITÉ LIMITÉE": "SARL", "SARL": "SARL", "AG": "AG", "AKTIENGESELLSCHAFT": "AG", "GESELLSCHAFT MIT BESCHRÄNKTER HAFTUNG": "GMBH", "GMBH": "GMBH", "SP Z OO": "SPZOO", "OAO": "OAO", "ZAO": "ZAO", "SA": "SA", "BV": "BV", "NV": "NV", "OY": "OY", "AS": "AS", "AB": "AB", "SRO": "SRO", "KK": "KK", "KABUSHIKI KAISHA": "KK","LIMITD": "LTD", "LIMTIED": "LTD", "LIMTIED.": "LTD", "LTD,": "LTD", "INC,": "INC"}
STATE_MAPPING = {"ALABAMA": "AL", "AL": "AL", "ALASKA": "AK", "AK": "AK", "ARIZONA": "AZ", "AZ": "AZ", "ARKANSAS": "AR", "AR": "AR", "CALIFORNIA": "CA", "CA": "CA", "COLORADO": "CO", "CO": "CO", "CONNECTICUT": "CT", "CT": "CT", "DELAWARE": "DE", "DE": "DE", "FLORIDA": "FL", "FL": "FL", "GEORGIA": "GA", "GA": "GA", "HAWAII": "HI", "HI": "HI", "IDAHO": "ID", "ID": "ID", "ILLINOIS": "IL", "IL": "IL", "INDIANA": "IN", "IN": "IN", "IOWA": "IA", "IA": "IA", "KANSAS": "KS", "KS": "KS", "KENTUCKY": "KY", "KY": "KY", "LOUISIANA": "LA", "LA": "LA", "MAINE": "ME", "ME": "ME", "MARYLAND": "MD", "MD": "MD", "MASSACHUSETTS": "MA", "MA": "MA", "MICHIGAN": "MI", "MI": "MI", "MINNESOTA": "MN", "MN": "MN", "MISSISSIPPI": "MS", "MS": "MS", "MISSOURI": "MO", "MO": "MO", "MONTANA": "MT", "MT": "MT", "NEBRASKA": "NE", "NE": "NE", "NEVADA": "NV", "NV": "NV", "NEW HAMPSHIRE": "NH", "NH": "NH", "NEW JERSEY": "NJ", "NJ": "NJ", "NEW MEXICO": "NM", "NM": "NM", "NEW YORK": "NY", "NY": "NY", "NORTH CAROLINA": "NC", "NC": "NC", "NORTH DAKOTA": "ND", "ND": "ND", "OHIO": "OH", "OH": "OH", "OKLAHOMA": "OK", "OK": "OK", "OREGON": "OR", "OR": "OR", "PENNSYLVANIA": "PA", "PA": "PA", "RHODE ISLAND": "RI", "RI": "RI", "SOUTH CAROLINA": "SC", "SC": "SC", "SOUTH DAKOTA": "SD", "SD": "SD", "TENNESSEE": "TN", "TN": "TN", "TEXAS": "TX", "TX": "TX", "UTAH": "UT", "UT": "UT", "VERMONT": "VT", "VT": "VT", "VIRGINIA": "VA", "VA": "VA", "WASHINGTON": "WA", "WA": "WA", "WEST VIRGINIA": "WV", "WV": "WV", "WISCONSIN": "WI", "WI": "WI", "WYOMING": "WY", "WY": "WY"}

### LISTS
jurisdictions = ["USA", "UNITED STATES", "UNITED STATES OF AMERICA","GB", "UK", "UNITED KINGDOM", "BRITAIN", "ENGLAND", "SCOTLAND", "WALES", "NORTHERN IRELAND","CA", "CANADA","MX", "MEXICO","DE", "GERMANY", "DEUTSCHLAND","FR", "FRANCE","IT", "ITALY", "ITALIA","ES", "SPAIN", "ESPANA", "ESPAÑA","PT", "PORTUGAL","IE", "IRELAND", "EIRE","NL", "NETHERLANDS", "HOLLAND", "NEDERLAND","BE", "BELGIUM", "BELGIE", "BELGIQUE","LU", "LUXEMBOURG", "LUX","CH", "SWITZERLAND", "SCHWEIZ", "SUISSE", "SVIZZERA","AT", "AUSTRIA", "ÖSTERREICH", "OSTERREICH","SE", "SWEDEN", "SVERIGE","NO", "NORWAY", "NORGE","DK", "DENMARK", "DANMARK","FI", "FINLAND", "SUOMI","IS", "ICELAND", "ISLAND","GR", "GREECE", "HELLAS", "ELLADA","TR", "TURKEY", "TURKIYE","CZ", "CZECH REPUBLIC", "CZECHIA", "CESKO","SK", "SLOVAKIA", "SLOVENSKO","PL", "POLAND", "POLSKA","SI", "SLOVENIA", "SLOVENIJA","HU", "HUNGARY", "MAGYARORSZAG", "MAGYARORSZÁG","CL", "CHILE","CO", "COLOMBIA","CR", "COSTA RICA","IL", "ISRAEL","JP", "JAPAN", "JPN", "NIPPON","KR", "KOREA", "SOUTH KOREA", "REPUBLIC OF KOREA", "KOREA REPUBLIC OF","AU", "AUSTRALIA", "AUS","NZ", "NEW ZEALAND", "N Z","KY", "CAYMAN ISLANDS", "CAYMAN","VG", "BVI", "BRITISH VIRGIN ISLANDS", "VIRGIN ISLANDS BRITISH", "VIRGIN ISLANDS UK","JE", "JERSEY","GG", "GUERNSEY","IM", "ISLE OF MAN", "MAN ISLAND","GI", "GIBRALTAR","BM", "BERMUDA","BS", "BAHAMAS","BB", "BARBADOS","SC", "SEYCHELLES","MU", "MAURITIUS","PA", "PANAMA","LI", "LIECHTENSTEIN","MC", "MONACO","AD", "ANDORRA","AI", "ANGUILLA","AW", "ARUBA","WS", "SAMOA","VU", "VANUATU","KN", "ST KITTS", "SAINT KITTS","VC", "ST VINCENT", "SAINT VINCENT","LC", "ST LUCIA", "SAINT LUCIA","AG", "ANTIGUA","BZ", "BELIZE","CK", "COOK ISLANDS","AN", "NETHERLANDS ANTILLES","CW", "CURACAO","HK", "HONG KONG","SG", "SINGAPORE","CN", "CHINA","IN", "INDIA","RU", "RUSSIA",]
company_monikers = ["LTD", "CORP", "INC", "CO", "PTY LTD", "PTY", "PT", "PTE", "PTE LTD", "UL", "SNC", "KD", "SPA", "PLC", "LP", "GP","LLC", "LLP", "PVT LTD", "PARTNERSHIP", "TRUST", "FUND", "FOUNDATION", "SA", "SARL", "AG", "GMBH", "SPZOO","OAO", "ZAO", "BV", "NV", "OY", "AS", "AB", "SRO", "KK"]
company_suffixes = ["LTD", "LTD.", "LIMTED", "LIMITED", "LIMTIED", "LIMTIED.", "LIMITD", "LTD,", "INC", "INC.", "INC,", "INCORPORATED", "INCORPORATD", "CORPORATION", "CORP", "CORP.","COMPANY", "CO", "CO.", "LP", "L.P.", "LIMITED PARTNERSHIP", "LTD PARTNERSHIP", "GP", "G.P.", "GENERAL PARTNERSHIP", "PARTNERSHIP", "LLC", "LLC.", "LIMITED LIABILITY COMPANY", "LIMITED LIABILITY CO", "LLP", "LLP.", "LIMITED LIABILITY PARTNERSHIP","TRUST", "FUND", "FOUNDATION", "HOLDINGS", "HLDGS", "HOLDING","SA", "S.A.", "SOCIETE ANONYME", "SOCIÉTÉ ANONYME", "SOCIEDAD ANONIMA", "SOCIETA ANONIMA", "SOCIÉTÉ ANONYME À RESPONSABILITÉ LIMITÉE", "SARL", "SOCIETE ANONYME A RESPONSABILITE LIMITEE", "GMBH", "GESELLSCHAFT MIT BESCHRÄNKTER HAFTUNG", "AKTIENGESELLSCHAFT", "AG", "SP Z OO", "SPZOO", "OAO", "ZAO", "BV", "NV", "OY", "AS", "AB", "SRO", "SRL", "SPA", "KD", "SNC", "KK", "KABUSHIKI KAISHA", "N.V.", "B.V.", "PTY LTD", "PTY. LTD.", "PROPRIETARY LIMITED", "PROPRIETARY", "PTE", "PTE LTD", "PTE. LTD.", "PT","UNLIMITED", "UNLTD", "UL", "PRIVATE LIMITED", "PVT LTD", "PVT. LTD.", "BVI", "BRITISH VIRGIN ISLANDS", "CAYMAN", "CAYMAN ISLANDS", "BERMUDA", "CURACAO", "ANGUILLA", "LIECHTENSTEIN", "MAURITIUS", "GUERNSEY", "JERSEY", "ISLE OF MAN", "MAN ISLAND", "VIRGIN ISLANDS", "VIRGIN ISLANDS BRITISH", "DEL", "S DE RL", "SDFRL", "KFT", "OOO", "EURL", "EI", ]
states = ["ALABAMA", "ALASKA", "ARIZONA", "ARKANSAS", "CALIFORNIA", "COLORADO", "CONNECTICUT", "DELAWARE", "FLORIDA", "GEORGIA", "HAWAII", "IDAHO", "ILLINOIS", "INDIANA", "IOWA", "KANSAS", "KENTUCKY", "LOUISIANA", "MAINE", "MARYLAND", "MASSACHUSETTS", "MICHIGAN", "MINNESOTA", "MISSISSIPPI", "MISSOURI", "MONTANA", "NEBRASKA", "NEVADA", "NEW HAMPSHIRE", "NEW JERSEY", "NEW MEXICO", "NEW YORK", "NORTH CAROLINA", "NORTH DAKOTA", "OHIO", "OKLAHOMA", "OREGON", "PENNSYLVANIA", "RHODE ISLAND", "SOUTH CAROLINA", "SOUTH DAKOTA", "TENNESSEE", "TEXAS", "UTAH", "VERMONT", "VIRGINIA", "WASHINGTON", "WEST VIRGINIA", "WISCONSIN", "WYOMING"]
state_isos = [" AL ", " AK ", " AZ ", " AR ", " CA ", " CO ", " CT ", " DE ", " FL ", " GA ", " HI ", " ID ", " IL "," IN ", " IA ", " KS ", " KY ", " LA ", " ME ", " MD ", " MA ", " MI ", " MN ", " MS ", " MO ", " MT "," NE ", " NV ", " NH ", " NJ ", " NM ", " NY ", " NC ", " ND ", " OH ", " OK ", " OR ", " PA ", " RI "," SC ", " SD ", " TN ", " TX ", " UT ", " VT ", " VA ", " WA ", " WV ", " WI ", " WY "]
canadian_provinces_territories = ["ALBERTA", "BRITISH COLUMBIA", "MANITOBA", "NEW BRUNSWICK", "NEWFOUNDLAND AND LABRADOR", "NEWFOUNDLAND", "LABRADOR", "NOVA SCOTIA", "NORTHWEST TERRITORIES", "NUNAVUT", "ONTARIO", "PRINCE EDWARD ISLAND", "P E I", "PEI", "QUEBEC", "QUÉBEC", "SASKATCHEWAN", "YUKON", "YUKON TERRITORY"]

#### Is US (?)
this method will determine if a line entry is in the United States via searching through American states, with a catch for Canada (CA) and California (CA)

In [4]:
def __is_US(entry):
    padded_entry = f" {entry} "
    match = next((prov for prov in canadian_provinces_territories if re.search(rf"\b{re.escape(prov.strip())}\b", padded_entry)), None)
    if match:
        return " CA"
    match_us = next((state for state in states if re.search(rf"\b{re.escape(state.strip())}\b", padded_entry)), None)
    if match_us:
        return f" {match_us.strip()} US"
    return ""

#### Replace Phrase Mapping With Space

This method simply adds a space when using other methods to mitigate combining county/state isos

In [5]:
def __replace_phrase_mapping_with_space(text, mapping):
    keys_sorted = sorted(mapping, key=lambda x: -len(x))
    pattern = r'|'.join(rf'\b{re.escape(k)}\b' for k in keys_sorted)
    def repl(match):
        return mapping[match.group(0)] + ' '
    return re.sub(pattern, repl, text)

#### Remove adjacent repeates

This is a handler to help remove if a method is incidentially used twice

In [6]:
def __remove_adjacent_repeats(entry):
    pattern = r'\b(\w{2,})( \1\b)+'
    return re.sub(pattern, r'\1', entry, flags=re.IGNORECASE)

#### Precleaning

this method employs various previous methods and mappings to "clean" the data

In [7]:
def __precleaning(entry):
    entry = entry.upper()
    entry = entry.replace(",", " ")
    entry = re.sub(r' +', ' ', entry)
    entry = re.sub(r'[^A-Z0-9\s]', '', entry)
    isus = __is_US(entry)
    if isus and not entry.endswith(isus):
        entry = entry + " " + isus

    entry = __replace_phrase_mapping_with_space(entry, COMPANY_MONIKERS)
    for name_variant, iso_code in sorted(STATE_MAPPING.items(), key=lambda x: -len(x[0])):
        entry = re.sub(rf'\b{name_variant}\b', f' {iso_code} ', entry) 
    for name_variant, iso_code in COUNTRY_MAPPING.items():
        entry = re.sub(rf'\b{name_variant}\b', f' {iso_code} ', entry)

    tokens = entry.split()
    tokens = [COMPANY_MONIKERS.get(tok, tok) for tok in tokens]
    tokens = [STATE_MAPPING.get(tok, tok) for tok in tokens]
    tokens = [COUNTRY_MAPPING.get(tok, tok) for tok in tokens]
    entry = ' '.join(tokens)
    entry = re.sub(r' +', ' ', entry).strip()
    entry = __remove_adjacent_repeats(entry)
    entry = re.sub(r'\bUSA\b', 'US', entry)
    return entry

#### Find Country

This method is used to isolate the country

In [8]:
def __find_country(text):
    for country in pycountry.countries:
        if country.name.upper() in text.upper():
            return country.alpha_2
        if hasattr(country, "official_name") and country.official_name.upper() in text.upper():
            return country.alpha_2
        if country.alpha_2 in text or country.alpha_3 in text:
            return country.alpha_2
    return None

#### Preprocess Recipients

this method is to extract certain relevent data points and put them in parts of a dictionary

In [9]:
def __preprocessRecipients(entity_str):
    entity_str = entity_str.strip()
    country_guess = entity_str[-2:].upper()
    country_obj = pycountry.countries.get(alpha_2=country_guess)
    country = None
    if country_obj:
        country = country_obj.alpha_2
        entity_wo_country = entity_str[:-2].strip()
    else:
        country = None
        entity_wo_country = entity_str
        for c in pycountry.countries:
            if c.alpha_2 in entity_str.upper():
                country = c.alpha_2
                break
            if c.name.upper() in entity_str.upper():
                country = c.alpha_2
                break
    
    # company moniker extraction
    match_moniker = next((moniker for moniker in company_monikers if moniker in entity_wo_country), None)
    if match_moniker:
        idx = entity_wo_country.find(match_moniker) + len(match_moniker)
        entityname = entity_wo_country[:idx]
        entity_wo_country_entity = entity_wo_country[idx:].lstrip()
    else:
        entityname = entity_wo_country
        entity_wo_country_entity = entity_wo_country

    # state extraction
    match_state = next((state_i for state_i in state_isos if state_i in entity_wo_country_entity), None)
    if match_state:
        state = match_state.strip()
        entity_wo_country_entity_state = entity_wo_country_entity.replace(match_state, ' ')
        entity_wo_country_entity_state = re.sub(r' +', ' ', entity_wo_country_entity_state).strip()
    else:
        state = None
        entity_wo_country_entity_state = entity_wo_country_entity

    # zip extraction
    match = re.search(r" (\d{5})\b", entity_wo_country_entity_state)
    if match:
        zip_code = match.group(1)
        entity_wo_country_entity_state_zip = (entity_wo_country_entity_state[:match.start()] + entity_wo_country_entity_state[match.end():]).strip()
    else:
        zip_code = None
        entity_wo_country_entity_state_zip = entity_wo_country_entity_state

    address_and_city = entity_wo_country_entity_state_zip

    return {
        "entity_name": entityname,
        "address_and_city": address_and_city,
        "state": state,
        "zip": zip_code,
        "country": country
    }

#### Expand Entity Fields

Calls helper preprocessRecipients with a stripped string

In [10]:
def __expand_entity_fields(entity_str):
    return __preprocessRecipients(entity_str.strip())

#### Add entity IDs

this method is the first dealing with ER (Entity Resoution) and tries to resolve entities with the most accurate portions of each enitities dictionary (i.e. first is an exact match for the entity name, second is a Levenshtein for the entity name, then a country comparison, etc).

Groups by the same real-world entity under a common cluster ID. This method uses the Splink library to perform deduplication via blocking rules and fuzzy comparisons on entity fields (name, country, state, address). Records that are predicted to match above a given probability threshold are clustered together, and each entity is assigned the cluster ID of its group as its `entity_id`.

Prediction threshold is set to 0.70 (match probability) for candidate generation. Clustering threshold is set to 0.95 (match probability) for group assignment. Splink parameters (m and u values) are not trained here, so I used default which may reduce accuracy.

**note for the future** use `linker.training` on relevent projects

In [11]:
def __add_entity_ids(entities):
    df = pd.DataFrame(entities)
    df["unique_id"] = df.index.astype(str)

    comparisons = [
        cl.ExactMatch("entity_name"),
        cl.LevenshteinAtThresholds("entity_name"),
        cl.ExactMatch("country"),
        cl.ExactMatch("state"),
        cl.ExactMatch("address_and_city"),
    ]

    blocking_rules = [
        "l.entity_name = r.entity_name",
        "(l.country = r.country) and (l.state = r.state)",
        "(l.entity_name = r.entity_name and l.country = r.country)",
        "levenshtein(l.entity_name, r.entity_name) < 2",
    ]

    settings = {
        "link_type": "dedupe_only",
        "unique_id_column_name": "unique_id",
        "blocking_rules_to_generate_predictions": blocking_rules,
        "comparisons": comparisons
    }

    db_api = DuckDBAPI()
    linker = Linker(df, settings, db_api)

    df_pred = linker.inference.predict(threshold_match_probability=0.7)
    cluster_df = linker.clustering.cluster_pairwise_predictions_at_threshold(df_pred, threshold_match_probability=0.95).as_pandas_dataframe()
    cluster_dict = cluster_df.set_index("entity_name")["cluster_id"].to_dict()

    for e in entities:
        e["entity_id"] = cluster_dict.get(e["entity_name"])
        
    return entities

#### Display waterfall chart

The waterfall chart shows how Splink arrives at a final match score for each pair of records, broken down comparison by comparison (e.g. here we are breaking it down via `entity_name`, `country`, `state`, and `address_and_city`), each comparison (`entity_name` [exact], `entity_name` [Levenshtein], `country`, `state`, `address_and_city`) either adds or subtracts from the match score depending on whether the fields agreed

In [12]:
def __display_waterfall_chart(entities):
    df = pd.DataFrame(entities)
    df["unique_id"] = df.index.astype(str)

    comparisons = [
        cl.ExactMatch("entity_name"),
        cl.LevenshteinAtThresholds("entity_name"),
        cl.ExactMatch("country"),
        cl.ExactMatch("state"),
        cl.ExactMatch("address_and_city"),
    ]

    blocking_rules = [
        "l.entity_name = r.entity_name",
        "(l.country = r.country) and (l.state = r.state)",
        "(l.entity_name = r.entity_name and l.country = r.country)",
        "levenshtein(l.entity_name, r.entity_name) < 2",
    ]

    settings = {
        "link_type": "dedupe_only",
        "unique_id_column_name": "unique_id",
        "blocking_rules_to_generate_predictions": blocking_rules,
        "comparisons": comparisons,
        "retain_intermediate_calculation_columns": True,
        "retain_matching_columns": True
    }

    db_api = DuckDBAPI()
    linker = Linker(df, settings, db_api)

    df_pred = linker.inference.predict(threshold_match_probability=0.7)
    cluster_df = linker.clustering.cluster_pairwise_predictions_at_threshold(df_pred, threshold_match_probability=0.95).as_pandas_dataframe()
    records = df_pred.as_pandas_dataframe().to_dict(orient="records")
    cluster_dict = cluster_df.set_index("entity_name")["cluster_id"].to_dict()
    chart = linker.visualisations.waterfall_chart(records, filter_nulls=False)
    return chart, cluster_df.set_index("entity_name")["cluster_id"].to_dict()

#### Evaluation analysis

this needs to be implemented, see [here](https://moj-analytical-services.github.io/splink/demos/tutorials/07_Evaluation.html)

In [13]:
def __evaluation_analysis(linker, labels_table):
    results = linker.evaluation.accuracy_analysis_from_labels_table(
        labels_table,
        output_type="threshold_selection",
        add_metrics=["f1"]
    )
    
    chart = results
    return chart

# Using Previous Methods to Analyze Provided "Messy" Data

In [14]:
sender = [
'John Doe 123 Green Lane Los Angeles, CA 90210',
'J Smith 33 Woodbridge St New York NY',
'ACME INVESTMENTS LTD BVI',
'Acme Investments Limited (B.V.I.)',
'Ryt Generale SRL',
'TYCHO FEEDER FUND INC NY USA',
'Acme Investments Limited Road Town Tortola VG',
'Jane Cyrus 384 Brandy Way Orlando FL USA',
'Bank of Cyprus,Strovolos,CY',
'AC ME TRUST INCORPORAT ED'
]

print(sender)

['John Doe 123 Green Lane Los Angeles, CA 90210', 'J Smith 33 Woodbridge St New York NY', 'ACME INVESTMENTS LTD BVI', 'Acme Investments Limited (B.V.I.)', 'Ryt Generale SRL', 'TYCHO FEEDER FUND INC NY USA', 'Acme Investments Limited Road Town Tortola VG', 'Jane Cyrus 384 Brandy Way Orlando FL USA', 'Bank of Cyprus,Strovolos,CY', 'AC ME TRUST INCORPORAT ED']


In [15]:
sender = [__precleaning(entry) for entry in sender]
sender = [__expand_entity_fields(entry) for entry in sender]
chart, cluster_dict = __display_waterfall_chart(sender)
sender = __add_entity_ids(sender)
display(chart)
df = pd.DataFrame(sender)
display(df)

Blocking time: 0.01 seconds
Predict time: 0.06 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'entity_name':
    m values not fully trained
Comparison: 'entity_name':
    u values not fully trained
Comparison: 'entity_name':
    m values not fully trained
Comparison: 'entity_name':
    u values not fully trained
Comparison: 'country':
    m values not fully trained
Comparison: 'country':
    u values not fully trained
Comparison: 'state':
    m values not fully trained
Comparison: 'state':
    u values not fully trained
Comparison: 'address_and_city':
    m values not fully trained
Comparison: 'address_and_city':
    u values not fully trained
The 'probability_two_random_records_match' setting has been set to the default value (0.0001). 
If this is not the desired be

alt.LayerChart(...)

,entity_name,address_and_city,state,zip,country,entity_id
0,JOHN DOE 123 GREEN LANE LOS ANGELES CA 90210,JOHN DOE 123 GREEN LANE LOS ANGELES,CA,90210,CA,0
1,J SMITH 33 WOODBRIDGE ST NY,J SMITH 33 WOODBRIDGE ST NY,NaN,NaN,US,1
2,ACME INVESTMENTS LTD,,NaN,NaN,VG,2
3,ACME INVESTMENTS LTD,,NaN,NaN,VG,2
4,RYT GENERALE SRL,RYT GENERALE SRL,NaN,NaN,AL,4
5,TYCHO FEEDER FUND INC,NY,NaN,NaN,US,5
6,ACME INVESTMENTS LTD,ROAD TOWN TORTOLA,NaN,NaN,VG,2
7,JANE CYRUS 384 BRANDY WAY ORLANDO FL,JANE CYRUS 384 BRANDY WAY ORLANDO FL,NaN,NaN,US,7
8,BANK OF CYPRUS STROVOLOS,BANK OF CYPRUS STROVOLOS,NaN,NaN,CY,8
9,AC ME TRUST INCORP,ORAT ED,NaN,NaN,AT,9


# Data Generation Portion Entities

In [16]:
def get_next_data_folder(base="data_"):
    existing = [d for d in os.listdir('.') if os.path.isdir(d) and re.match(rf'^{base}\d+$', d)]
    if not existing:
        n = 0
    else:
        nums = [int(re.search(rf'{base}(\d+)', x).group(1)) for x in existing]
        n = max(nums) + 1
    folder = f"{base}{n}"
    os.makedirs(folder, exist_ok=True)
    return folder

This method adds random line breaks etc. to mess with the name to make ER (Entity Resolution) more difficault

In [17]:
def random_misspell_name(name):
    if random.random() < 0.1: ## 10% chance to misspell random thing or have a character break in name
        i = random.randint(0, len(name)-1)
        c = name[i]
        if c.isalpha():
            typo = random.choice([chr(((ord(c)-65+random.randint(1,2))%26)+65) for _ in range(1)])
            return name[:i]+typo+name[i+1:]
    return name

Generate sub Entities This generates the fake data for specific companies/people/banks

In [18]:
def generate_sub_entities(choice, is_peak, peak):
    sub_e = []
    if choice == 'individual':
        for _ in range(random.randrange(1, 6)):
            sender = f"{fake.name()} {fake.building_number()} {fake.street_name()} {fake.city()} {fake.state_abbr()} {fake.postcode()}"
            sub_e.append(sender)
        
    elif choice == 'company':
        entity_name = fake.company()
        if is_peak:
            j = 0
            while j < peak:
                rand = random.randrange(1, 4)
                if rand == 1:
                    suffix = random.choice(company_suffixes)
                    juris = random.choice(jurisdictions)
                    address = f"{fake.building_number()} {fake.street_name()}"
                    sender = f"{entity_name} {suffix} {address} {juris}"
                    j = j + 1
                    sub_e.append(sender)
                elif (rand == 2 or rand == 3):
                    sender = entity_name
                    j = j + 1
                    sub_e.append(sender)
                elif (rand == 4):
                    juris = random.choice(jurisdictions)
                    sender = f"{entity_name} {juris}"
                    j = j + 1
                    sub_e.append(sender)
        else:
            rand = random.randrange(1, 4)
            j = 0
            while j < rand:
                rand = random.randrange(1, 4)
                if rand == 1:
                    suffix = random.choice(company_suffixes)
                    juris = random.choice(jurisdictions)
                    address = f"{fake.building_number()} {fake.street_name()}"
                    sender = f"{entity_name} {suffix} {address} {juris}"
                    j = j + 1
                    sub_e.append(sender)
                elif (rand == 2 or rand == 3):
                    sender = entity_name
                    j = j + 1
                    sub_e.append(sender)
                elif (rand == 4):
                    juris = random.choice(jurisdictions)
                    sender = f"{entity_name} {juris}"
                    j = j + 1
                    sub_e.append(sender)

    else:
        bank_name = f"{fake.company()} Bank"
        if is_peak:
            j = 0
            while j < peak:
                if (random.choice([True, False])):
                    sender = bank_name
                    j = j + 1   
                    sub_e.append(sender)
                else:
                    juris = random.choice(jurisdictions)
                    sender = f"{bank_name} {juris}"
                    j = j + 1
                    sub_e.append(sender)
        else:
            rand = random.randrange(1, 4)
            j = 0
            while j < peak:
                if (random.choice([True, False])):
                    sender = bank_name
                    j = j + 1   
                    sub_e.append(sender)
                else:
                    juris = random.choice(jurisdictions)
                    sender = f"{bank_name} {juris}"
                    j = j + 1
                    sub_e.append(sender)
    return sub_e

Generate Entities according to number of entities, peak, and number of peaks

In [19]:
def generate_entities(n, p, np):
    e = []
    i = 0
    peaks = 0

    global dist_e
    dist_e = 0

    while i < n:
        choice = random.choices(['individual', 'company', 'bank'], weights=[0.3, 0.6, 0.1])[0]
        if peaks < np:
            is_peak = random.choices([True, False], weights=[0.02, 0.98])[0]
        else:
            is_peak = False

        if choice == 'individual':
            entities_sub = generate_sub_entities(choice, False, p)
            incr = len(entities_sub)
            dist_e = dist_e + 1
            for x in range(incr):
                e.append(entities_sub[x])
            i = i + incr

        elif choice == 'company':
            entities_sub = generate_sub_entities(choice, is_peak, p)
            incr = len(entities_sub)
            dist_e = dist_e + 1
            for x in range(incr):
                e.append(entities_sub[x])
            i = i + incr

        else:
            entities_sub = generate_sub_entities(choice, is_peak, p)
            incr = len(entities_sub)
            dist_e = dist_e + 1
            for x in range(incr):
                e.append(entities_sub[x])
            i = i + incr

    return e    

prints out DF to .csv

In [20]:
def send_to_csv(df, fx):
    __current_data_folder__ = get_next_data_folder()
    df.to_csv(f"./{__current_data_folder__}/{fx}.csv", index=False)

In [21]:
fake = Faker()
Faker.seed(42)
random.seed(42)

n = 10000 ## 10,000 entities currently
peak = 300
num_peaks = 2

entities = generate_entities(n, peak, num_peaks)
print("Number of entieties: ", len(entities))

Number of entieties:  10074


In [22]:
processed_entities = [__precleaning(entry) for entry in entities]
processed_entities = [__expand_entity_fields(entry) for entry in processed_entities]
processed_entities = __add_entity_ids(processed_entities)

df_entities__f = pd.DataFrame(processed_entities)
send_to_csv(df_entities__f, "synthetic_etities")
num_unique = df_entities__f['entity_id'].nunique()
print("unique entities: " + str(num_unique) + ", expected unique entities: " + str(dist_e))
display(df_entities__f)

Blocking time: 37.96 seconds
Predict time: 0.64 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'entity_name':
    m values not fully trained
Comparison: 'entity_name':
    u values not fully trained
Comparison: 'entity_name':
    m values not fully trained
Comparison: 'entity_name':
    u values not fully trained
Comparison: 'country':
    m values not fully trained
Comparison: 'country':
    u values not fully trained
Comparison: 'state':
    m values not fully trained
Comparison: 'state':
    u values not fully trained
Comparison: 'address_and_city':
    m values not fully trained
Comparison: 'address_and_city':
    u values not fully trained
The 'probability_two_random_records_match' setting has been set to the default value (0.0001). 
If this is not the desired b

unique entities: 704, expected unique entities: 245


,entity_name,address_and_city,state,zip,country,entity_id
0,RODRIGUEZ FIGUEROA AND SANCHEZ LLC,18196 ANTHONY FORGE,NaN,NaN,NL,0
1,WAGNER INC,PTE 386 SHANE HARBORS,NaN,NaN,AU,1
2,ABIGAIL SHAFFER 654 JAS,ON TRACK CURTISFURT,CT,47553,AF,2
3,LINDSEY ROMAN 781 MILLER HILL BARBARALAND AZ 8...,LINDSEY ROMAN 781 MILLER HILL BARBARALAND,AZ,87174,AL,3
4,MEJIA I,MEJIA I,NaN,NaN,NC,4
...,...,...,...,...,...,...
10069,HALE INC,1859 ARNOLD REST,NaN,NaN,IT,10069
10070,HALE I,HALE I,NaN,NaN,NC,10000
10071,HALE I,HALE I,NaN,NaN,NC,10000
10072,HALE I,HALE I,NaN,NaN,NC,10000


In [23]:
entity_counts = df_entities__f['entity_id'].value_counts()
valid_entity_ids = entity_counts[(entity_counts >= 5) & (entity_counts <= 55)].index

if len(valid_entity_ids) > 0:
    chosen_entity_id = valid_entity_ids[0]
    filtered_entities = df_entities__f[df_entities__f['entity_id'] == chosen_entity_id]
    print(filtered_entities)
else:
    print("No entity_id found with between 15 and 45 entries.")

chart, cluster_dict = __display_waterfall_chart(filtered_entities)
display(chart)

Blocking time: 0.00 seconds
Predict time: 0.06 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'entity_name':
    m values not fully trained
Comparison: 'entity_name':
    u values not fully trained
Comparison: 'entity_name':
    m values not fully trained
Comparison: 'entity_name':
    u values not fully trained
Comparison: 'country':
    m values not fully trained
Comparison: 'country':
    u values not fully trained
Comparison: 'state':
    m values not fully trained
Comparison: 'state':
    u values not fully trained
Comparison: 'address_and_city':
    m values not fully trained
Comparison: 'address_and_city':
    u values not fully trained
The 'probability_two_random_records_match' setting has been set to the default value (0.0001). 
If this is not the desired be

                    entity_name        address_and_city state  zip country  \
4806  BROWN JENSEN AND RICE LTD         925 ERIC SPRING   NaN  NaN      VC   
4847  BROWN JENSEN AND RICE LTD  83388 DUNN ROAD ISLAND   NaN  NaN      AD   
4854  BROWN JENSEN AND RICE LTD        3940 PENA BRANCH   NaN  NaN      IT   
4945  BROWN JENSEN AND RICE LTD        211 JASMIN ROADS   NaN  NaN      JP   
4967  BROWN JENSEN AND RICE LTD      994 KNAPP CAUSEWAY   NaN  NaN      GB   
4974  BROWN JENSEN AND RICE LTD        68147 DAVID WALL   NaN  NaN      JP   
4999  BROWN JENSEN AND RICE LTD      6990 JAMES VILLAGE   NaN  NaN      CZ   
5006  BROWN JENSEN AND RICE LTD          588 JARED SPUR   NaN  NaN      VG   
5017  BROWN JENSEN AND RICE LTD   7438 THOMPSON LANDING   NaN  NaN      SI   
5024  BROWN JENSEN AND RICE LTD      57845 KAYLA COURTS   NaN  NaN      IS   

     entity_id  
4806      5024  
4847      5024  
4854      5024  
4945      5024  
4967      5024  
4974      5024  
4999      5024  
5006 

alt.LayerChart(...)